<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《Build a Large Language Model From Scratch》</a> 一书的补充代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 从零实现字节对编码（BPE）Tokenizer

- 这是一个独立 Notebook，用于从零实现流行的字节对编码（BPE）分词算法；这类算法用于 GPT-2 到 GPT-4、Llama 3 等模型，这里主要用于教学目的
- 关于分词目的的更多细节，请参考[第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)；这里的代码是解释 BPE 算法的 bonus 材料
- OpenAI 为训练原始 GPT 模型实现的原始 BPE tokenizer 可以在[这里](https://github.com/openai/gpt-2/blob/master/src/encoder.py)找到
- BPE 算法最早在 1994 年由 Philip Gage 的论文 "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)" 中提出
- 由于计算性能更好，现在包括 Llama 3 在内的大多数项目会使用 OpenAI 开源的 [tiktoken 库](https://github.com/openai/tiktoken)；例如，它可以加载预训练的 GPT-2 和 GPT-4 tokenizer（Llama 3 模型也使用 GPT-4 tokenizer 训练）
- 上述实现和本 Notebook 中实现的区别之一是：这里还包含一个用于训练 tokenizer 的函数（用于教学目的）
- 还有一个名为 [minBPE](https://github.com/karpathy/minbpe) 的实现也支持训练，性能可能更好（这里的实现重点是教学）；与 `minbpe` 不同，我的实现还额外允许加载原始 OpenAI tokenizer 的词表和 BPE “merges”（此外，Hugging Face tokenizers 也能训练和加载多种 tokenizer；更多信息可见一位读者在[这个 GitHub discussion](https://github.com/rasbt/LLMs-from-scratch/discussions/485) 中训练尼泊尔语 BPE tokenizer 的例子）

&nbsp;
# 1. 字节对编码（BPE）的核心思想

- BPE 的核心思想是把文本转换成整数表示（token ID），以便用于 LLM 训练（参见[第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="1000px">

&nbsp;
## 1.1 比特和字节

- 在进入 BPE 算法之前，先介绍字节这个概念
- 可以考虑把文本转换成字节数组（毕竟 BPE 中的 B 就是 "byte"）：

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")#bytearray()是一个内置函数，用于创建一个可变的字节数组。它接受一个字符串和一个编码格式作为参数，将字符串转换为字节数组。
print(byte_ary)

bytearray(b'This is some text')


- 当我们对 `bytearray` 对象调用 `list()` 时，每个字节都会被当成一个单独元素，结果是一个由对应字节值组成的整数列表：

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- 这确实是一种把文本转换成 token ID 表示的有效方式，而这种表示正是 LLM 嵌入层所需要的
- 不过，这种方法的缺点是会为每个字符创建一个 ID（短文本也会产生很多 ID！）
- 也就是说，对于一个 17 个字符的输入文本，我们必须使用 17 个 token ID 作为 LLM 的输入：

In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- 如果你之前用过 LLM，可能知道 BPE tokenizer 的词表中通常有完整单词或子词的 token ID，而不是每个字符一个 token ID
- 例如，GPT-2 tokenizer 会把同一段文本（"This is some text"）分成 4 个 token，而不是 17 个：`1212, 318, 617, 2420`
- 你可以使用交互式 [tiktoken app](https://tiktokenizer.vercel.app/?model=gpt2) 或 [tiktoken 库](https://github.com/openai/tiktoken) 进行复查：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="1000px">


In [3]:
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# 打印 [1212, 318, 617, 2420]

[1212, 318, 617, 2420]

- 由于一个字节由 8 个比特组成，因此单个字节可以表示 2<sup>8</sup> = 256 种可能的值，范围是 0 到 255
- 你可以执行 `bytearray(range(0, 257))` 来确认这一点；它会提示 `ValueError: byte must be in range(0, 256)`
- BPE tokenizer 通常把这 256 个值作为最开始的 256 个单字符 token；可以运行下面的代码直观检查：

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
打印：
0: !
1: "
2: #
...
255: �  # <---- 到这里为止是单字符 token
256:  t
257:  a
...
298: ent
299:  n
"""
```

- 注意上面第 256 和 257 项不是单字符值，而是双字符值（一个空白字符 + 一个字母）；这是原始 GPT-2 BPE tokenizer 的一个小缺点（GPT-4 tokenizer 已经改进了这一点）

In [4]:
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")

0: !
1: "
2: #
3: $
4: %
5: &
6: '
7: (
8: )
9: *
10: +
11: ,
12: -
13: .
14: /
15: 0
16: 1
17: 2
18: 3
19: 4
20: 5
21: 6
22: 7
23: 8
24: 9
25: :
26: ;
27: <
28: =
29: >
30: ?
31: @
32: A
33: B
34: C
35: D
36: E
37: F
38: G
39: H
40: I
41: J
42: K
43: L
44: M
45: N
46: O
47: P
48: Q
49: R
50: S
51: T
52: U
53: V
54: W
55: X
56: Y
57: Z
58: [
59: \
60: ]
61: ^
62: _
63: `
64: a
65: b
66: c
67: d
68: e
69: f
70: g
71: h
72: i
73: j
74: k
75: l
76: m
77: n
78: o
79: p
80: q
81: r
82: s
83: t
84: u
85: v
86: w
87: x
88: y
89: z
90: {
91: |
92: }
93: ~
94: �
95: �
96: �
97: �
98: �
99: �
100: �
101: �
102: �
103: �
104: �
105: �
106: �
107: �
108: �
109: �
110: �
111: �
112: �
113: �
114: �
115: �
116: �
117: �
118: �
119: �
120: �
121: �
122: �
123: �
124: �
125: �
126: �
127: �
128: �
129: �
130: �
131: �
132: �
133: �
134: �
135: �
136: �
137: �
138: �
139: �
140: �
141: �
142: �
143: �
144: �
145: �
146: �
147: �
148: �
149: �
150: �
151: �
152: �
153: �
154: �
155: �
156: �
157: �
158:

&nbsp;
## 1.2 构建词表

- BPE 分词算法的目标是构建一个常见子词词表，例如 `298: ent`（例如可以出现在 *entangle, entertain, enter, entrance, entity, ...* 中），甚至完整单词，例如：

```
318: is
617: some
1212: This
2420: text
```

- BPE 算法最早在 1994 年由 Philip Gage 的论文 "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)" 中提出
- 在进入实际代码实现之前，可以把如今用于 LLM tokenizer 的形式总结为下面几节描述的内容。

&nbsp;
## 1.3 BPE 算法概要

**1. 找出高频符号对**
- 在每次迭代中扫描文本，找出最常出现的一对字节（或字符）

**2. 替换并记录**

- 用一个新的占位 ID 替换这对符号（这个 ID 尚未使用；例如如果从 0...255 开始，第一个占位 ID 可以是 256）
- 在查找表中记录这个映射
- 查找表的大小是一个超参数，也叫“词表大小”（GPT-2 的词表大小是 50,257）

**3. 重复，直到没有收益**

- 不断重复第 1 步和第 2 步，持续合并最频繁的符号对
- 当无法进一步压缩时停止（例如没有任何符号对出现超过一次）

**解压缩（解码）**

- 为了恢复原始文本，使用查找表把每个 ID 替换回对应的符号对，从而反向执行这个过程


&nbsp;
## 1.4 BPE 算法示例

### 1.4.1 编码部分的具体示例（1.3 节中的第 1 步和第 2 步）

- 假设训练数据文本是 `the cat in the hat`，我们想基于它为 BPE tokenizer 构建词表

**迭代 1**

1. 找出高频符号对
  - 在这段文本中，"th" 出现了两次（一次在开头，一次在第二个 "e" 前面）

2. 替换并记录
  - 用一个尚未使用的新 token ID 替换 "th"，例如 256
  - 新文本变成：`<256>e cat in <256>e hat`
  - 新词表为

```
  0: ...
  ...
  256: "th"
```

**迭代 2**

1. **找出高频符号对**  
   - 在文本 `<256>e cat in <256>e hat` 中，符号对 `<256>e` 出现了两次

2. **替换并记录**  
   - 用一个尚未使用的新 token ID 替换 `<256>e`，例如 `257`。  
   - 新文本变成：
     ```
     <257> cat in <257> hat
     ```
   - 更新后的词表为：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**迭代 3**

1. **找出高频符号对**  
   - 在文本 `<257> cat in <257> hat` 中，符号对 `<257> ` 出现了两次（一次在开头，一次在 “hat” 前面）。

2. **替换并记录**  
   - 用一个尚未使用的新 token ID 替换 `<257> `，例如 `258`。  
   - 新文本变成：
     ```
     <258>cat in <258>hat
     ```
   - 更新后的词表为：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- 以此类推

&nbsp;
### 1.4.2 解码部分的具体示例（1.3 节中的第 3 步）

- 为了恢复原始文本，我们按这些 token ID 被引入的相反顺序，将每个 token ID 替换回对应的符号对
- 从最终压缩后的文本开始：`<258>cat in <258>hat`
- 替换 `<258>` → `<257> `：`<257> cat in <257> hat`  
- 替换 `<257>` → `<256>e`：`<256>e cat in <256>e hat`
- 替换 `<256>` → "th"：`the cat in the hat`

&nbsp;
## 2. 一个简单的 BPE 实现

- 下面用一个 Python 类实现上面描述的算法，并模拟 `tiktoken` 的 Python 用户接口
- 注意，上面的编码部分描述的是通过 `train()` 完成的原始训练步骤；不过 `encode()` 方法的工作方式类似（只是因为要处理特殊 token，所以看起来稍微复杂一些）：

1. 将输入文本拆分成单个字节
2. 反复查找并替换（合并）相邻 token（符号对）；如果它们匹配已学习到的 BPE merge，就按从高到低的优先级（也就是学习到的顺序）进行合并
3. 持续合并，直到不能再应用新的 merge
4. 最终得到的 token ID 列表就是编码输出

In [3]:
from collections import Counter, deque
from functools import lru_cache
import re
import json


class BPETokenizerSimple:
    def __init__(self):
        # 将 token_id 映射到 token_str（例如 {11246: "some"}）
        self.vocab = {}
        # 将 token_str 映射到 token_id（例如 {"some": 11246}）
        self.inverse_vocab = {}
        # BPE merges 字典：{(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

        # 对于官方 OpenAI GPT-2 merges，使用 rank 字典：
        # 形式为 {(string_A, string_B): rank}，rank 越小优先级越高
        self.bpe_ranks = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # 使用与 encode() 相同的边界规则对训练文本做预分词
        tokens = self.pretokenize_text(text)

        # 用唯一字符初始化词表，如果存在 "Ġ" 也包含进去
        # 从前 256 个 ASCII 字符开始
        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(
            char for char in sorted({char for token in tokens for char in token})
            if char not in unique_chars
        )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # 添加允许的特殊 token
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # 将每个预分词片段分词为字符 ID
        token_id_sequences = [
            [self.inverse_vocab[char] for char in token]
            for token in tokens
        ]

        # BPE 第 1-3 步：反复查找并替换高频符号对
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_id_sequences, mode="most")
            if pair_id is None:
                break
            token_id_sequences = self.replace_pair(token_id_sequences, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id

        # 用合并后的 token 构建词表
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def load_vocab_and_merges_from_openai(self, vocab_path, bpe_merges_path):
        """
        Load pre-trained vocabulary and BPE merges from OpenAI's GPT-2 files.

        Args:
            vocab_path (str): Path to the vocab file (GPT-2 calls it 'encoder.json').
            bpe_merges_path (str): Path to the bpe_merges file  (GPT-2 calls it 'vocab.bpe').
        """
        # 加载词表
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            # encoder.json 是 {token_str: id}；这里需要 id->str 和 str->id
            self.vocab = {int(v): k for k, v in loaded_vocab.items()}
            self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}
    
        # ID 198 必须是 GPT-2 的可打印换行字符 'Ċ'（U+010A）
        if "Ċ" not in self.inverse_vocab or self.inverse_vocab["Ċ"] != 198:
            raise KeyError("Vocabulary missing GPT-2 newline glyph 'Ċ' at id 198.")
    
        # ID 50256 必须是 <|endoftext|>
        if "<|endoftext|>" not in self.inverse_vocab or self.inverse_vocab["<|endoftext|>"] != 50256:
            raise KeyError("Vocabulary missing <|endoftext|> at id 50256.")
    
        # 为 '\n' -> 198 提供一个方便的别名
        # 在 vocab 中保留可打印字符 'Ċ'，这样 BPE merges 才能继续工作
        if "\n" not in self.inverse_vocab:
            self.inverse_vocab["\n"] = self.inverse_vocab["Ċ"]

        if "\r" not in self.inverse_vocab:
            if 201 in self.vocab:
                self.inverse_vocab["\r"] = 201
            else:
                raise KeyError("Vocabulary missing carriage return token at id 201.")

        # 加载 GPT-2 merges 并保存 rank
        self.bpe_ranks = {}
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            lines = file.readlines()
            if lines and lines[0].startswith("#"):
                lines = lines[1:]
    
            rank = 0
            for line in lines:
                token1, *rest = line.strip().split()
                if len(rest) != 1:
                    continue
                token2 = rest[0]
                if token1 in self.inverse_vocab and token2 in self.inverse_vocab:
                    self.bpe_ranks[(token1, token2)] = rank
                    rank += 1
                else:
                    # 如果符号不在词表中，可以安全跳过这些 pair
                    pass


    def encode(self, text, allowed_special=None):
        """
        Encode the input text into a list of token IDs, with tiktoken-style handling of special tokens.
    
        Args:
            text (str): The input text to encode.
            allowed_special (set or None): Special tokens to allow passthrough. If None, special handling is disabled.
    
        Returns:
            List of token IDs.
        """
    
        # ---- 本节用于在 allowed special tokens 方面模拟 tiktoken ----
        specials_in_vocab = [
            tok for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>")
        ]
        if allowed_special is None:
            # 不允许任何特殊 token
            disallowed = [tok for tok in specials_in_vocab if tok in text]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        else:
            # 允许某些特定 token（例如这里用于 <|endoftext|>）
            disallowed = [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        # -----------------------------------------------------------------------------

        token_ids = []
        # 如果允许某些特殊 token，就围绕它们切分文本，并直接传递对应 ID
        if allowed_special is not None and len(allowed_special) > 0:
            special_pattern = "(" + "|".join(
                re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)
            ) + ")"
    
            last_index = 0
            for match in re.finditer(special_pattern, text):
                prefix = text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None))  # encode prefix normally
    
                special_token = match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token {special_token} not found in vocabulary.")
                last_index = match.end()
    
            text = text[last_index:]  # remainder to process normally
    
            # 额外防护：处理可能残留的其他特殊 token 字面量
            disallowed = [
                tok for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
            ]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

    
        # ---- 换行符和回车符处理 ----
        tokens = self.pretokenize_text(text)
        # ---------------------------------------------------------------
    
        # 将 token 映射为 ID（必要时使用 BPE）
        for tok in tokens:
            if tok in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[tok])
            else:
                token_ids.extend(self.tokenize_with_bpe(tok))
    
        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # 将 token 分成单个字符（作为初始 token ID）
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        # 如果尚未加载 OpenAI GPT-2 merges，则使用这里的训练式做法
        if not self.bpe_ranks:
            can_merge = True
            while can_merge and len(token_ids) > 1:
                can_merge = False
                new_tokens = []
                i = 0
                while i < len(token_ids) - 1:
                    pair = (token_ids[i], token_ids[i + 1])
                    if pair in self.bpe_merges:
                        merged_token_id = self.bpe_merges[pair]
                        new_tokens.append(merged_token_id)
                        # 为了教学目的，可以取消下面的注释：
                        # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                        i += 2  # Skip the next token as it's merged
                        can_merge = True
                    else:
                        new_tokens.append(token_ids[i])
                        i += 1
                if i < len(token_ids):
                    new_tokens.append(token_ids[i])
                token_ids = new_tokens
            return token_ids

        # 否则，使用 rank 执行 GPT-2 风格的合并：
        # 1）将 token_ids 转回每个 ID 对应的字符串“符号”
        symbols = [self.vocab[id_num] for id_num in token_ids]

        # 反复合并 rank 最小的符号对的所有出现位置
        while True:
            # 收集所有相邻符号对
            pairs = set(zip(symbols, symbols[1:]))
            if not pairs:
                break

            # 找到 rank 最好（最小）的符号对
            min_rank = float("inf")
            bigram = None
            for p in pairs:
                r = self.bpe_ranks.get(p, float("inf"))
                if r < min_rank:
                    min_rank = r
                    bigram = p

            # 如果不存在有效的 ranked pair，就结束
            if bigram is None or bigram not in self.bpe_ranks:
                break

            # 合并该符号对的所有出现位置
            first, second = bigram
            new_symbols = []
            i = 0
            while i < len(symbols):
                # 如果在位置 i 看到 (first, second)，就合并它们
                if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
                    new_symbols.append(first + second)  # merged symbol
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols

            if len(symbols) == 1:
                break

        # 最后，将合并后的符号转回 ID
        merged_ids = [self.inverse_vocab[sym] for sym in symbols]
        return merged_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        out = []
        for tid in token_ids:
            if tid not in self.vocab:
                raise ValueError(f"Token ID {tid} not found in vocab.")
            tok = self.vocab[tid]

            # 将 GPT-2 特殊字符映射回真实字符
            if tid == 198 or tok == "\n":
                out.append("\n")
            elif tid == 201 or tok == "\r":
                out.append("\r")
            elif tok.startswith("Ġ"):
                out.append(" " + tok[1:])
            else:
                out.append(tok)
        return "".join(out)

    def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Save the vocabulary and BPE merges to JSON files.

        Args:
            vocab_path (str): Path to save the vocabulary.
            bpe_merges_path (str): Path to save the BPE merges.
        """
        # 保存词表
        with open(vocab_path, "w", encoding="utf-8") as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        # 将 BPE merges 保存为字典列表
        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id}
                           for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)

    def load_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Load the vocabulary and BPE merges from JSON files.

        Args:
            vocab_path (str): Path to the vocabulary file.
            bpe_merges_path (str): Path to the BPE merges file.
        """
        # 加载词表
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(k): v for k, v in loaded_vocab.items()}
            self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}

        # 加载 BPE merges
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            merges_list = json.load(file)
            for merge in merges_list:
                pair = tuple(merge["pair"])
                new_id = merge["new_id"]
                self.bpe_merges[pair] = new_id

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def pretokenize_text(text):
        tokens = []
        parts = re.split(r'(\r\n|\r|\n)', text)
        for part in parts:
            if part == "":
                continue
            if part == "\r\n":
                tokens.append("\r")
                tokens.append("\n")
                continue
            if part == "\r":
                tokens.append("\r")
                continue
            if part == "\n":
                tokens.append("\n")
                continue

            # 没有换行的普通片段：
            # - 如果单词前有空格，就给第一个单词加上 'Ġ' 前缀，并且
            #   为额外空格添加独立的 'Ġ'
            # - 如果片段末尾有空格（例如换行前），则添加
            #   独立的 'Ġ' token（tiktoken 会为 'Ġ' 产生 ID 220）
            pending_spaces = 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces += len(m.group(1))
                else:
                    word = m.group(2)
                    if pending_spaces > 0:
                        for _ in range(pending_spaces - 1):
                            tokens.append("Ġ")  # remaining spaces as standalone
                        tokens.append("Ġ" + word) # one leading space
                        pending_spaces = 0
                    else:
                        tokens.append(word)
            # 尾随空格（后面没有单词）：添加独立的 'Ġ' token
            for _ in range(pending_spaces):
                tokens.append("Ġ")
        return tokens

    @staticmethod
    def find_freq_pair(token_id_sequences, mode="most"):
        pairs = Counter(
            pair
            for token_ids in token_id_sequences
            for pair in zip(token_ids, token_ids[1:])
        )

        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_id_sequences, pair_id, new_id):
        replaced_sequences = []

        for token_ids in token_id_sequences:
            dq = deque(token_ids)
            replaced = []

            while dq:
                current = dq.popleft()
                if dq and (current, dq[0]) == pair_id:
                    replaced.append(new_id)
                    # 移除这对符号中的第 2 个 token，第 1 个已经被替换/移除
                    dq.popleft()
                else:
                    replaced.append(current)

            replaced_sequences.append(replaced)

        return replaced_sequences

- 上面的 `BPETokenizerSimple` 类包含很多代码，在这个 Notebook 里详细讨论它超出了范围；不过下一节会简要介绍用法，帮助你更好地理解这些类方法

## 3. BPE 实现 walkthrough

- 实际使用中，我强烈建议使用 [tiktoken](https://github.com/openai/tiktoken)，因为上面的实现侧重可读性和教学目的，而不是性能
- 不过它的用法和 tiktoken 大体类似，只是 tiktoken 没有训练方法
- 下面通过几个例子看看上面的 `BPETokenizerSimple` Python 代码如何工作（详细代码讨论不在本 Notebook 范围内）

### 3.1 训练、编码和解码

- 首先，把一些示例文本作为训练数据集：

In [1]:
import os
import requests

def download_file_if_absent(url, filename, search_dirs):
    for directory in search_dirs:
        file_path = os.path.join(directory, filename)
        if os.path.exists(file_path):
            print(f"{filename} already exists in {file_path}")
            return file_path

    target_path = os.path.join(search_dirs[0], filename)
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(target_path, "wb") as out_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    out_file.write(chunk)
        print(f"Downloaded {filename} to {target_path}")
    except Exception as e:
        print(f"Failed to download {filename}. Error: {e}")

    return target_path


verdict_path = download_file_if_absent(
    url=(
         "https://raw.githubusercontent.com/rasbt/"
         "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
         "the-verdict.txt"
    ),
    filename="the-verdict.txt",
    search_dirs=["ch02/01_main-chapter-code/", "../01_main-chapter-code/", "."]
)

with open(verdict_path, "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

the-verdict.txt already exists in ch02/01_main-chapter-code/the-verdict.txt


- 接下来，用词表大小 1,000 初始化并训练 BPE tokenizer
- 注意，由于前面讨论过的字节值，词表大小默认已经是 256；因此这里只“学习”744 个词表条目（如果考虑 `<|endoftext|>` 特殊 token 和 `Ġ` 空白 token，准确来说是 742 个）
- 作为对比，GPT-2 词表有 50,257 个 token，GPT-4 词表有 100,256 个 token（tiktoken 中的 `cl100k_base`），GPT-4o 使用 199,997 个 token（tiktoken 中的 `o200k_base`）；相比上面的简单示例文本，它们都有大得多的训练集

In [4]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- 你可能想检查一下词表内容（但注意这会生成一个很长的列表）

In [6]:
print(tokenizer.vocab)
print(len(tokenizer.vocab))

{0: '\x00', 1: '\x01', 2: '\x02', 3: '\x03', 4: '\x04', 5: '\x05', 6: '\x06', 7: '\x07', 8: '\x08', 9: '\t', 10: '\n', 11: '\x0b', 12: '\x0c', 13: '\r', 14: '\x0e', 15: '\x0f', 16: '\x10', 17: '\x11', 18: '\x12', 19: '\x13', 20: '\x14', 21: '\x15', 22: '\x16', 23: '\x17', 24: '\x18', 25: '\x19', 26: '\x1a', 27: '\x1b', 28: '\x1c', 29: '\x1d', 30: '\x1e', 31: '\x1f', 32: ' ', 33: '!', 34: '"', 35: '#', 36: '$', 37: '%', 38: '&', 39: "'", 40: '(', 41: ')', 42: '*', 43: '+', 44: ',', 45: '-', 46: '.', 47: '/', 48: '0', 49: '1', 50: '2', 51: '3', 52: '4', 53: '5', 54: '6', 55: '7', 56: '8', 57: '9', 58: ':', 59: ';', 60: '<', 61: '=', 62: '>', 63: '?', 64: '@', 65: 'A', 66: 'B', 67: 'C', 68: 'D', 69: 'E', 70: 'F', 71: 'G', 72: 'H', 73: 'I', 74: 'J', 75: 'K', 76: 'L', 77: 'M', 78: 'N', 79: 'O', 80: 'P', 81: 'Q', 82: 'R', 83: 'S', 84: 'T', 85: 'U', 86: 'V', 87: 'W', 88: 'X', 89: 'Y', 90: 'Z', 91: '[', 92: '\\', 93: ']', 94: '^', 95: '_', 96: '`', 97: 'a', 98: 'b', 99: 'c', 100: 'd', 101: 'e'

- 这个词表是通过 742 次合并创建的（`= 1000 - len(range(0, 256)) - len(special_tokens) - "Ġ" = 1000 - 256 - 1 - 1 = 742`）

In [ ]:
print(len(tokenizer.bpe_merges))
print(tokenizer.bpe_merges)


742
{(256, 116): 258, (104, 101): 259, (256, 97): 260, (105, 110): 261, (256, 104): 262, (256, 115): 263, (256, 119): 264, (256, 111): 265, (258, 259): 266, (111, 117): 267, (114, 101): 268, (105, 116): 269, (256, 109): 270, (105, 115): 271, (101, 100): 272, (97, 116): 273, (110, 100): 274, (256, 98): 275, (261, 103): 276, (256, 112): 277, (97, 115): 278, (101, 110): 279, (258, 111): 280, (256, 100): 281, (256, 102): 282, (256, 259): 283, (101, 114): 284, (262, 97): 285, (256, 73): 286, (256, 108): 287, (265, 102): 288, (111, 110): 289, (256, 99): 290, (258, 104): 291, (45, 45): 292, (101, 115): 293, (111, 114): 294, (101, 108): 295, (264, 278): 296, (256, 269): 297, (97, 110): 298, (260, 274): 299, (275, 101): 300, (97, 114): 301, (105, 99): 302, (105, 109): 303, (265, 110): 304, (97, 99): 305, (111, 119): 306, (256, 261): 307, (262, 271): 308, (117, 114): 309, (256, 101): 310, (256, 103): 311, (108, 101): 312, (256, 110): 313, (118, 101): 314, (115, 116): 315, (111, 109): 316, (117, 

- 这意味着前 256 个条目是单字符 token

- 接下来，用 `encode` 方法使用已经创建的 merges 来编码一些文本：

In [14]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)
decoded_text = tokenizer.decode(token_ids)
print(decoded_text)

[74, 361, 310, 109, 98, 420, 397, 100, 300, 428, 116, 121, 519, 699, 299, 808, 534]
Jack embraced beauty through art and life.


In [17]:
input_text = "Jack embraced beauty through art and life.<|endoftext|> "
token_ids = tokenizer.encode(input_text, allowed_special={"<|endoftext|>"})
print(token_ids)
decoded_text = tokenizer.decode(token_ids)
print(decoded_text)

[74, 361, 310, 109, 98, 420, 397, 100, 300, 428, 116, 121, 519, 699, 299, 808, 534, 257, 256]
Jack embraced beauty through art and life.<|endoftext|> 


In [18]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 56
Number of token IDs: 19


- 从上面的长度可以看到，一个 42 字符的句子被编码成了 20 个 token ID；与基于字符/字节的编码相比，输入长度大约减半了

- 注意，词表本身会在 `decode()` 方法中使用，从而把 token ID 映射回文本：

In [19]:
print(token_ids)

[74, 361, 310, 109, 98, 420, 397, 100, 300, 428, 116, 121, 519, 699, 299, 808, 534, 257, 256]


In [20]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.<|endoftext|> 


- 遍历每个 token ID，可以帮助我们更好地理解这些 token ID 如何通过词表被解码：

In [21]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

74 -> J
361 -> ack
310 ->  e
109 -> m
98 -> b
420 -> ra
397 -> ce
100 -> d
300 ->  be
428 -> au
116 -> t
121 -> y
519 ->  through
699 ->  art
299 ->  and
808 ->  lif
534 -> e.
257 -> <|endoftext|>
256 ->  


- 可以看到，大多数 token ID 表示 2 字符子词；这是因为训练数据文本很短，重复词不多，而且我们使用的词表大小相对较小

- 总结一下，调用 `decode(encode())` 应该能够重现任意输入文本：

In [22]:
tokenizer.decode(
    tokenizer.encode("This is some text.")
)

'This is some text.'

In [23]:
tokenizer.decode(
    tokenizer.encode("This is some text with \n newline characters.")
)

'This is some text with \n newline characters.'

### 3.2 保存和加载 tokenizer

- 接下来，看一下如何保存训练好的 tokenizer，方便后续复用：

In [24]:
# 保存训练好的 tokenizer
tokenizer.save_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

In [25]:
# 加载 tokenizer
tokenizer2 = BPETokenizerSimple()
tokenizer2.load_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

- 加载后的 tokenizer 应该能够产生和之前相同的结果：

In [26]:
print(tokenizer2.decode(token_ids))

Jack embraced beauty through art and life.<|endoftext|> 


In [27]:
tokenizer2.decode(
    tokenizer2.encode("This is some text with \n newline characters.")
)

'This is some text with \n newline characters.'

&nbsp;
### 3.3 从 OpenAI 加载原始 GPT-2 BPE tokenizer

- 最后，加载 OpenAI 的 GPT-2 tokenizer 文件

In [28]:
# 如果这些文件尚不存在于当前目录，则下载它们

# 定义要搜索的目录和要下载的文件
search_directories = ["ch02/02_bonus_bytepair-encoder/gpt2_model/", "../02_bonus_bytepair-encoder/gpt2_model/", "."]

files_to_download = {
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe": "vocab.bpe",
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json": "encoder.json"
}

# 确保目录存在，并在需要时下载文件
paths = {}
for url, filename in files_to_download.items():
    paths[filename] = download_file_if_absent(url, filename, search_directories)

vocab.bpe already exists in ch02/02_bonus_bytepair-encoder/gpt2_model/vocab.bpe
encoder.json already exists in ch02/02_bonus_bytepair-encoder/gpt2_model/encoder.json


- 接下来，通过 `load_vocab_and_merges_from_openai` 方法加载这些文件：

In [29]:
tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=paths["encoder.json"], bpe_merges_path=paths["vocab.bpe"]
)

- 词表大小应该是 `50257`，可以通过下面的代码确认：

In [30]:
len(tokenizer_gpt2.vocab)

50257

- 现在可以通过我们的 `BPETokenizerSimple` 对象使用 GPT-2 tokenizer：

In [31]:
input_text = "This is some text"
token_ids = tokenizer_gpt2.encode(input_text)
print(token_ids)

[1212, 318, 617, 2420]


In [32]:
print(tokenizer_gpt2.decode(token_ids))

This is some text


- 你可以使用交互式 [tiktoken app](https://tiktokenizer.vercel.app/?model=gpt2) 或 [tiktoken 库](https://github.com/openai/tiktoken) 再次确认它产生了正确的 token：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# 打印 [1212, 318, 617, 2420]
```

&nbsp;
# 4. 结论

- 就是这样！这就是 BPE 的基本工作方式，并且包含一个用于创建新 tokenizer 的训练方法，也可以从 OpenAI 原始 GPT-2 模型中加载 GPT-2 tokenizer 词表和 merges
- 希望这个简短教程对学习有所帮助；如果有问题，欢迎在[这里](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a)开启新的 Discussion
- 如果想和其他 tokenizer 实现做性能对比，请查看[这个 Notebook](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb)